In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm

from pycircstat.tests import *
import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson

# Boilerplate

1,2,13,24,

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,25,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5np.save
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
def bandpower_relative(data, sf, band, window_sec=None, relative=False):
    """Compute the average power of the signal x in a specific frequency band.

    Parameters
    ----------
    data : 1d-array
        Input signal in the time-domain.
    sf : float
        Sampling frequency of the data.
    band : list
        Lower and upper frequencies of the band of interest.
    window_sec : float
        Length of each window in seconds.
        If None, window_sec = (1 / min(band)) * 2
    relative : boolean
        If True, return the relative power (= divided by the total power of the signal).
        If False (default), return the absolute power.

    Return
    ------
    bp : float
        Absolute or relative band power.
    """

    band = np.asarray(band)
    low, high = band

    psd, freqs = mne.time_frequency.psd_array_multitaper(data, sf, fmin=low, fmax=high, adaptive=True, low_bias=True, normalization='full', verbose=False)

    difference = np.diff(freqs)
    freq_res = np.mean(difference)
    idx_band = np.logical_and(freqs >= low, freqs <= high)

    integral_band = simpson(psd[idx_band], dx=freq_res)
    average_band = np.mean(psd[idx_band])

    # Compute power for frequencies between 2Hz and 45Hz
    psd_total, freqs_total = mne.time_frequency.psd_array_multitaper(data, sf, fmin=2, fmax=45, adaptive=True, low_bias=True, normalization='full', verbose=False)
    difference_total = np.diff(freqs_total)
    freq_res_total = np.mean(difference_total)
    idx_band = np.logical_and(freqs_total >= 2, freqs_total <= 45)
    integral_total = simpson(psd_total, dx=freq_res_total)
    average_total = np.mean(psd_total)
    # Compute the relative power of the given frequency band
    integral_relative = integral_band / integral_total
    average_relative = average_band / average_total


    return integral_relative, average_relative

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

In [ ]:
cfg = load_config()
cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")


freq_bands = {"delta": (2, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}



all_subject_power_data_integral = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
all_subject_power_data_average = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}

In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:


def process_subject_data(cfg, subject_index, dir_path, freq_bands, device, all_subject_relevant_channels = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}):
    #cfg.dataset.data_directory = dir_path
    #cfg.dataset.subject_index = subject_index
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    #cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    #cli_args = parse_args()
    #cfg = update_config(cfg, cli_args)
    #save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:,:,:900]

    all_trials_integral = []
    all_trials_average = []
    for trial in range(all_epochs.shape[0]):
        all_data_trial_integral = {k: {ch:[] for ch in ch_names} for k in freq_bands.keys()}
        all_data_trial_average = {k: {ch:[] for ch in ch_names} for k in freq_bands.keys()}

        for idx, ch in enumerate(ch_names):
            data = all_epochs[trial, idx, :]
            for band_name, band_range in freq_bands.items():
                integral, average = bandpower_relative(data, 1000, band_range)
                all_data_trial_integral[band_name][ch].append(integral)
                all_data_trial_average[band_name][ch].append(average)

        all_trials_integral.append(all_data_trial_integral)
        all_trials_average.append(all_data_trial_average)
        
    return all_trials_integral, all_trials_average



In [ ]:
def plot_power_amplitude(all_subject_power_data, all_subject_amplitude_data,subject_index=2 , show_plot=True):
    all_trials = all_subject_power_data[subject_index]
    all_amplitudes = all_subject_amplitude_data[subject_index]
    #all_uncertainties = all_subject_uncertainty_data[subject_index]
    # data is of the shape subject_index, trial, band, channel
    # make separate plot for each channel and separate subplot for each frequency band where power is on the x-axis and amplitude is on the y-axis
    
    freq_bands = {  "delta": (0.5, 4),
                    "theta": (4, 8),
                    "alpha": (8, 12),
                    "beta": (12, 30),
                    "gamma": (30, 45)}
    ch_names = list(all_trials[0]["theta"].keys())
    high_correlations = {f: {} for f in freq_bands.keys()}
    corrs_abs = {f: {} for f in freq_bands.keys()}
    corrs = {f: {} for f in freq_bands.keys()}
    for band_idx, band_name in enumerate(freq_bands.keys()):
        corrs_abs[band_name] = {}
        corrs[band_name] = {}
        if show_plot:
            fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(10, 2), sharey=True)
            fig.suptitle(f"Channel: {ch}")
        for ch in ch_names:
            corrs_abs[band_name][ch] = {}
            corrs[band_name][ch] = {}
            current_data = []
            for trial in range(len(all_trials)):
                power = all_trials[trial][band_name][ch]
                current_data.append(power[0])
            # throw away trials with top 1% of power and corresponding amplitudes
            current_data = np.array(current_data)
            all_amplitudes = np.array(all_amplitudes)
            top_1_percent = np.percentile(current_data, 99)
            indices = current_data <= top_1_percent
            corr, pval = stats.pearsonr(current_data[indices], all_amplitudes[indices])
            corrs_abs[band_name][ch]["stat"] = np.abs(corr)
            corrs[band_name][ch]["stat"] = corr
            corrs_abs[band_name][ch]["pval"] = pval
            corrs[band_name][ch]["pval"] = pval


            if show_plot:
            
                axs[band_idx].set_xlabel('Power')
                axs[band_idx].set_ylabel('Amplitude')
                if np.abs(corr) >= 0.3:
                    axs[band_idx].set_title(f"{band_name}, corr: {corr:.2f}")
                    high_correlations[band_name][ch] = corr
                    axs[band_idx].scatter(current_data[indices], all_amplitudes[indices], alpha=0.5, c='k')

                else:
                    axs[band_idx].set_title(f"{band_name}")
                    axs[band_idx].scatter(current_data[indices], all_amplitudes[indices], alpha=0.5)
                fig.savefig(f"correlation_plots/channel_{ch}_subject_{subject_index}.png")
                plt.show()
    return high_correlations, corrs_abs, corrs

In [ ]:
def plot_power_amplitude_filtered(all_subject_power_data, all_subject_amplitude_data, subject_index=2, threshold=0.3, show_plot=True):
    all_trials = all_subject_power_data[subject_index]
    all_amplitudes = all_subject_amplitude_data[subject_index]
    
    freq_bands = {"theta": (4, 8),
                  "alpha": (8, 12),
                  "beta": (12, 30),
                  "gamma": (30, 45)}
    ch_names = list(all_trials[0]["theta"].keys())
    high_correlations = {f: {} for f in freq_bands.keys()}
    
    for ch in ch_names:
        for band_idx, band_name in enumerate(freq_bands.keys()):
            current_data = []
            for trial in range(len(all_trials)):
                power = all_trials[trial][band_name][ch]
                current_data.append(power[0])
            
            current_data = np.array(current_data)
            all_amplitudes = np.array(all_amplitudes)
            top_1_percent = np.percentile(current_data, 99)
            indices = current_data <= top_1_percent
            corr = np.corrcoef(current_data[indices], all_amplitudes[indices])[0, 1]
            
            if threshold == -1 or np.abs(corr) >= threshold:
                if show_plot:
                    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 3))
                    fig.suptitle(f"Subject {subject_index}, Channel: {ch}")
                    ax.set_xlabel('Power')
                    ax.set_ylabel('Amplitude')
                    ax.set_title(f"{band_name}, corr: {corr:.2f}", y=0.95)
                    ax.scatter(current_data[indices], all_amplitudes[indices], alpha=0.5, c='k')
                    os.makedirs("correlation_plots", exist_ok=True)
                    fig.savefig(f"correlation_plots/subject_{subject_index}_channel_{ch}_band_{band_name}.png")
                
                high_correlations[band_name][ch] = corr

    return high_correlations


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

In [ ]:
#def create_mne_info(ch_names, sfreq=1000):
#    montage = mne.channels.make_standard_montage('standard_1005')
#    ch_names = montage.ch_names[:n_channels]
#    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
#    info.set_montage(montage)
#    return info

# preprocess data (get power per frequency band and amplitudes for each trial)

# only execute for the fist time
cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")

#freq_bands = {"theta": (4, 8),
#              "alpha": (8, 12),
#              "beta": (12, 30),
#              "gamma": (30, 45)}

all_subject_power_data_integral = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
all_subject_power_data_average = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
for subject_index in cfg.dataset.test_subject_indices:
    all_trials_integral, all_trials_average = process_subject_data(cfg, subject_index, dir_path, freq_bands, device)
    all_subject_power_data_integral[subject_index] = all_trials_integral
    all_subject_power_data_average[subject_index] = all_trials_average

os.makedirs(dir_path, exist_ok=True)
np.save(os.path.join(dir_path, "delta_all_subject_power_relative_data_integral.npy"), all_subject_power_data_integral)
np.save(os.path.join(dir_path, "_delta_all_subject_power_relative_data_average.npy"), all_subject_power_data_average)

subject_ x channel x freq_band

save data after run!

In [ ]:
all_subject_power_data_integral = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
all_subject_power_data_average = {subject_index: [] for subject_index in cfg.dataset.test_subject_indices}
cfg = load_config()
cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")

freq_bands = {"delta": (2, 4),
              "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
  

all_subject_power_relative_data_integral = np.load(os.path.join(dir_path, "merged_all_subject_relative_power_data_integral.npy"), allow_pickle=True).item()
#all_subject_power_relative_data_average = np.load(os.path.join(dir_path, "all_subject_power_relative_data_average.npy"), allow_pickle=True).item()

# Create a merged dictionary with data from both delta and other frequency bands
merged_power_data = {}

# First copy all existing frequency bands data
for subject_index in cfg.dataset.test_subject_indices:
    merged_power_data[subject_index] = copy.deepcopy(all_subject_power_relative_data_integral[subject_index])

# Now add the delta band data
for subject_index in cfg.dataset.test_subject_indices:
    # Check if delta data exists for this subject
    if subject_index in all_subject_power_data_integral_delta:
        # For each trial, merge the delta data
        for trial_idx, trial_data in enumerate(all_subject_power_data_integral_delta[subject_index]):
            if trial_idx < len(merged_power_data[subject_index]):
                # Add delta band to the existing trial data
                merged_power_data[subject_index][trial_idx]['delta'] = trial_data['delta']

# Save the merged data
np.save(os.path.join(dir_path, "merged_all_subject_relative_power_data_integral.npy"), merged_power_data)

# Print summary
print(f"Merged data created for {len(merged_power_data)} subjects")
print(f"Available frequency bands: {list(merged_power_data[cfg.dataset.test_subject_indices[0]][0].keys())}")

In [ ]:
import pickle
all_subject_amplitude_data = {}
all_subject_uncertainty_data = {}
data_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject in cfg.dataset.test_subject_indices:
    with open(os.path.join(data_path, f"subject_{subject}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        all_subject_amplitude_data[subject] = data['predictions']
        all_subject_uncertainty_data[subject] = data['uncertainties']

# subject 2

## compare power per frequency band vs predicted amplitude

In [ ]:
plot_power_amplitude_filtered(all_subject_power_relative_data_integral, all_subject_amplitude_data, subject_index=2)

In [ ]:
plot_power_amplitude(all_subject_power_relative_data_integral, all_subject_amplitude_data, subject_index=2)

## get raw labels for every subject

In [ ]:

def get_raw_labels(subject_index):
    cfg = load_config()
    cwd = os.getcwd()
    dir_path = os.path.join(cwd, "frequency_power_data")
    cfg = load_config()
    cfg.dataset.data_directory = dir_path
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    return all_labels_raw[150:]
  

In [ ]:
cfg = load_config()
all_subjects_raw_labels = {}
for subject_index in cfg.dataset.test_subject_indices:
    all_subjects_raw_labels[subject_index] = get_raw_labels(subject_index)

## compare power vs actual amplitude

In [ ]:

plot_power_amplitude(all_subject_power_data_integral, all_subjects_raw_labels, subject_index=2)

# For all subjects

## compare power vs predicted labels

In [ ]:
all_subjects_highest_corrs = {subject_index: plot_power_amplitude_filtered(all_subject_power_relative_data_integral, all_subject_amplitude_data, subject_index=subject_index, show_plot=False, threshold=-1) for subject_index in cfg.dataset.test_subject_indices}


In [ ]:
all_subjects_highest_corrs_all = {subject_index: plot_power_amplitude(all_subject_power_relative_data_integral, all_subject_amplitude_data, subject_index=subject_index, show_plot=False)[2] for subject_index in cfg.dataset.test_subject_indices}
all_subjects_highest_corrs_all_abs = {subject_index: plot_power_amplitude(all_subject_power_relative_data_integral, all_subject_amplitude_data, subject_index=subject_index, show_plot=False)[1] for subject_index in cfg.dataset.test_subject_indices}



In [ ]:
all_subjects_highest_corrs_all

In [ ]:
all_subjects_highest_corrs_all_abs

In [ ]:
np.save(os.path.join(cwd, "all_subjects_highest_corrs_relative_power.npy"), all_subjects_highest_corrs_all)
np.save(os.path.join(cwd, "all_subjects_highest_corrs_abs_relative_power.npy"), all_subjects_highest_corrs_all_abs)

In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    plot_power_amplitude_filtered(all_subject_power_data_integral, all_subject_amplitude_data, subject_index=subject_index)

# check correlation between 10 most important channels pe subject and power amplitude correlations

## top channel functions

In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict,k=10):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += k-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")

freq_bands = {"theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}




In [ ]:
def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict,k=k)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]
    #top_k_channels_dict = {ch: sum_over_channel_points_dict[ch] for ch in top_k_channels}
    

    return top_k_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 
 

In [ ]:
#Dont execute!#
#for subject_index in cfg.dataset.test_subject_indices:
#    cfg.dataset.data_directory = dir_path
#    cfg.dataset.subject_index = subject_index
#    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
#    cli_args = parse_args()
#    cfg = update_config(cfg, cli_args)
#    save_config(cfg)
#    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
#    all_epochs = all_epochs[150:,:,:900]
#    gradshap = compute_gradshap(all_epochs, subject_index=subject_index)
#    np.save(os.path.join(dir_path, f"gradshap_subject_{subject_index}.npy"), gradshap)



## top 10 channels per subject

In [ ]:
cfg = load_config()
top_10_per_subject = []
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_10_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_10_per_subject.append(top_10_channels)

In [ ]:
top_10_per_subject

In [ ]:
top_10_per_subject

In [ ]:
def extract_top_k_channels_and_mean_corr(all_subjects_corrs,k=10):
    top_10_channels_per_subject = {}
    mean_corr_per_subject = {}
    
    for subject, freq_bands in all_subjects_corrs.items():
        top_10_channels_per_subject[subject] = {}
        mean_corr_per_subject[subject] = {}
        
        for band, channels in freq_bands.items():
            sorted_channels = sorted(channels.items(), key=lambda item: abs(item[1]), reverse=True)[:k]
            top_10_channels_per_subject[subject][band] = dict(sorted_channels)
            
            mean_corr = np.mean([abs(corr) for ch, corr in sorted_channels])
            mean_corr_per_subject[subject][band] = mean_corr
    
    return top_10_channels_per_subject, mean_corr_per_subject

top_10_channels_per_subject_abs, mean_corr_per_subject_abs = extract_top_k_channels_and_mean_corr(all_subjects_highest_corrs_all_abs,k=60)

In [ ]:
top_10_channels_per_subject, mean_corr_per_subject= extract_top_k_channels_and_mean_corr(all_subjects_highest_corrs_all,k=60)

In [ ]:
def compute_summary_top_10(top_10):
    # across subjects and channels compute the min value, max value, the mean of the absolute values and the standard deviation
    all_corrs = {band:[] for band in top_10[1].keys()}
    all_corrs_abs = {band:[] for band in top_10[1].keys()}
    for subject, bands in top_10.items():
        for band, channels in bands.items():
            for channel, value in channels.items():
                all_corrs[band].append(value)
                all_corrs_abs[band].append(abs(value))
    
    summary = {}
    for band in all_corrs.keys():
        summary[band] = {}
        summary[band]["min"] = np.nanmin(all_corrs[band])
        summary[band]["max"] = np.nanmax(all_corrs[band])
        summary[band]["mean"] = np.nanmean(all_corrs_abs[band])
        summary[band]["std"] = np.nanstd(all_corrs_abs[band])
    return summary
                
                
                

In [ ]:
compute_summary_top_10(top_10_channels_per_subject)

In [ ]:
np.save("top_10_relative_power.npy", top_10_channels_per_subject)

## for gamma channel (most relevant)

In [ ]:
def compare_channels(cfg, top_k_per_subject, all_subjects_highest_corrs, frequency_band='gamma'):
    subject_ratios = []
    subject_common_channels = []
    subject_sig_channels = []
    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print(f"Channels with high correlation in {frequency_band} band:", list(all_subjects_highest_corrs[subject_index][frequency_band].keys()))

        common_channels = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        num_common_channels = len(common_channels)
        num_significant_channels = len(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        denominator = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), len(all_subjects_highest_corrs[subject_index][frequency_band].keys())))
        ratio = num_common_channels / denominator
        subject_common_channels.append(num_common_channels)
        subject_ratios.append(ratio)
        subject_sig_channels.append(num_significant_channels)
        print(f"Number of common channels: {num_common_channels}")
        print(f"Ratio: {ratio:.2f}")
        print()
    
    empty_list_indices = subject_common_channels[subject_common_channels ==0]
    plt.bar(np.arange(len(subject_ratios)), subject_ratios)
    for idx, num_sig in enumerate(subject_sig_channels):
        if num_sig == 0:
            plt.plot(idx, subject_ratios[idx], 'ro')

    plt.xticks(np.arange(len(subject_ratios)), cfg.dataset.test_subject_indices)
    plt.xlabel('Subject Index')
    plt.ylabel('Ratio of Common Channels')
    plt.title(f'Common Channels Ratio for {frequency_band} Band')
    plt.savefig(f"common_channels_ratio_{frequency_band}_relative_power.png")
    plt.show()

# Example usage:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject_abs, frequency_band='gamma')



## for beta channel

In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject_abs, frequency_band='beta')

## for alpha channel

In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='alpha')

## for theta channel

In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='theta')

there definitely seems to be some correlation between the gamma band power, and the predicted amplitude for some channels&subjects as well as a correlation between power x amplitude correlation and most important channels

get top k-channels and check if there is a correlation with power amplitude correlation 
Potentially also check relative power

# rank correlation for all 60 channels

In [ ]:
cfg = load_config()
top_60_per_subject = []
top_60_per_subject_dict = {}
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_60_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 60, update_channel_points_linear)
    top_60_per_subject.append(top_60_channels)
    top_60_per_subject_dict[subject_index] = sum_over_channel_points_dict

In [ ]:
from scipy.stats import spearmanr
rank_correlations = {}

for freq_band in freq_bands.keys():
    rank_correlations[freq_band] = {}
    for subject_index in cfg.dataset.test_subject_indices:
        top_60_channels = top_60_per_subject_dict[subject_index]
        corrs = all_subjects_highest_corrs_all_abs[subject_index][freq_band]
        print(subject_index)

        # Compute the Spearman rank correlation
        
        rank_corr, pval = spearmanr(list(top_60_channels.values()), list(corrs.values()))
        print(f"Rank correlation: {rank_corr:.2f}, p-value: {pval:.2f}")
      
        
        rank_correlations[freq_band][subject_index] = (rank_corr, pval)

In [ ]:
np.save( "highest_corrs_relative_power.npy", all_subjects_highest_corrs_all) 

In [ ]:
np.save( "highest_corrs_relative_power_abs.npy", all_subjects_highest_corrs_all_abs)

In [ ]:
import matplotlib.pyplot as plt


# Plot rank correlations for each frequency band
for freq_band in freq_bands:
    plt.figure(figsize=(12, 6))
    subjects = list(rank_correlations[freq_band].keys())
    rank_corrs = [rank_correlations[freq_band][subject][0] for subject in subjects]
    p_values = [rank_correlations[freq_band][subject][1] for subject in subjects]

    plt.bar(subjects, rank_corrs, color='blue', alpha=0.7, label='Rank Correlation')
    plt.scatter(subjects, rank_corrs, c=['red' if p < (0.05/60) else 'black' for p in p_values], label='p < 0.05', zorder=5)
    
    plt.xlabel('Subject Index')
    plt.ylabel('Rank Correlation')
    plt.title(f'Rank Correlation for {freq_band} Band')
    plt.axhline(y=0, color='gray', linestyle='--')
    plt.legend()
    plt.show()